# 方向二 Colab Launcher：单细胞多组学调控与空间轨迹映射（对应 Paper 2 & 3）

本 Notebook 专为 **Google Colab Pro (`L4 24GB` + 建议开启 `High-RAM`)** 设计：
- **严守依赖隔离**：直接通过系统包 `bedtools` + 固定版本 `pip` 安装（`numpy<2`, `anndata<0.12`, `scanpy<1.12`），**不混装方向一的 `TRIDENT`/`OpenSlide`/`pyvips`**；
- **高速计算与持久同步**：数据下载与训练均在 `/content/` 本地 SSD 进行，同时通过 `run_logged()` 将完整终端输出与 traceback 实时写入 `MyDrive/MedDemo_backup/direction2/logs/`（供本地 Agent 直接读日志 Debug），每个阶段结束后自动把预处理对象、模型输出和图表备份到 Google Drive。

## Step 1：硬件/内存检查 + 挂载 Google Drive

In [ ]:
!nvidia-smi
!free -h
!df -h /content

from google.colab import drive
drive.mount('/content/drive')

## Step 2：配置仓库来源与日志/备份辅助函数

In [ ]:
import os
import subprocess
from pathlib import Path

# === 请按需修改以下参数 ===
GITHUB_REPO_URL = "https://github.com/WilliamWu9871/MedDemo.git"
GIT_BRANCH = "main"
DRIVE_CODE_DIR = "/content/drive/MyDrive/MedDemo"  # 仅当 GITHUB_REPO_URL 为空时使用

WORK_ROOT = Path("/content/MedDemo")
DIR2_ROOT = WORK_ROOT / "direction2_singlecell_spatial_multiomics"
SUB2A_ROOT = DIR2_ROOT / "subdemo_2a_multiome_egrn"
SUB2B_ROOT = DIR2_ROOT / "subdemo_2b_spatial_trajectory"
VENV_DIR = Path("/content/venv_meddemo_dir2_py311")

DRIVE_BACKUP_DIR = Path("/content/drive/MyDrive/MedDemo_backup/direction2")
DRIVE_LOG_DIR = DRIVE_BACKUP_DIR / "logs"
DRIVE_BACKUP_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_LOG_DIR.mkdir(parents=True, exist_ok=True)

if GITHUB_REPO_URL:
    if (WORK_ROOT / ".git").exists():
        subprocess.run(["git", "-C", str(WORK_ROOT), "pull", "origin", GIT_BRANCH], check=True)
    else:
        subprocess.run(["git", "clone", "-b", GIT_BRANCH, GITHUB_REPO_URL, str(WORK_ROOT)], check=True)
else:
    if not Path(DRIVE_CODE_DIR).exists():
        raise FileNotFoundError(f"请填写 GITHUB_REPO_URL 或将代码放到 {DRIVE_CODE_DIR}")
    WORK_ROOT.mkdir(parents=True, exist_ok=True)
    subprocess.run(["rsync", "-av", "--exclude", "*.pdf", f"{DRIVE_CODE_DIR}/", f"{WORK_ROOT}/"], check=True)

os.chdir(DIR2_ROOT)
print("Working dir:", os.getcwd())
print("Live logs dir (synced to Drive):", DRIVE_LOG_DIR)
print("Isolated Python 3.11 venv:", VENV_DIR)


def run_logged(cmd: str, log_name: str, cwd: Path = DIR2_ROOT, use_venv: bool = True) -> None:
    """Run shell command, stream output live, and flush full stdout/stderr to Google Drive."""
    log_path = DRIVE_LOG_DIR / log_name
    activate_script = VENV_DIR / "bin" / "activate"
    full_cmd = f"source '{activate_script}' && {cmd}" if (use_venv and activate_script.exists()) else cmd
    env = {**os.environ, "PYTHONUNBUFFERED": "1"}
    print(f"==> [{cwd.name}] Running: {cmd}\n==> Log file: {log_path}")
    with log_path.open("w", encoding="utf-8") as log_f:
        proc = subprocess.Popen(
            ["bash", "-lc", full_cmd],
            cwd=str(cwd),
            env=env,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        assert proc.stdout is not None
        for line in proc.stdout:
            print(line, end="")
            log_f.write(line)
            log_f.flush()
        ret = proc.wait()
    if ret != 0:
        raise RuntimeError(f"Command failed (exit={ret}). Full traceback saved at: {log_path}")


def backup_subdemo(sub_name: str) -> None:
    """Sync processed objects and outputs (figures/tables/checkpoints) to Google Drive."""
    sub_root = DIR2_ROOT / sub_name
    for rel in ["data/processed", "outputs"]:
        src = sub_root / rel
        if src.exists():
            dst = DRIVE_BACKUP_DIR / sub_name / rel
            dst.mkdir(parents=True, exist_ok=True)
            subprocess.run(["rsync", "-a", f"{src}/", f"{dst}/"], check=True)
    print(f"[Backup] Synced {sub_name} -> {DRIVE_BACKUP_DIR / sub_name}")


def restore_subdemo(sub_name: str) -> None:
    """Restore cached processed datasets/outputs from Google Drive after a reconnect."""
    sub_root = DIR2_ROOT / sub_name
    for rel in ["data/processed", "outputs"]:
        src = DRIVE_BACKUP_DIR / sub_name / rel
        if src.exists():
            dst = sub_root / rel
            dst.mkdir(parents=True, exist_ok=True)
            subprocess.run(["rsync", "-a", f"{src}/", f"{dst}/"], check=True)
    print(f"[Restore] Restored {sub_name} from {DRIVE_BACKUP_DIR / sub_name}")

## Step 3：创建 Python 3.11 隔离环境、安装 `bedtools` / `scverse` / `scDoRI` / `cell2location` 依赖，并克隆官方仓库
Colab 默认 Python 3.13 与 `bioFAM/scDoRI`（要求 `>=3.10,<3.13` 及 `numpy<2`）不兼容，因此本步使用 `uv` 自动创建 `/content/venv_meddemo_dir2_py311` 隔离环境，安装全部 CUDA 依赖并注册 Jupyter 内核 `meddemo_dir2_py311`（后续步骤及 Papermill Notebook 会自动激活并使用该内核，**无需重启 Runtime**）。

In [ ]:
run_logged(
    f"bash scripts/setup_colab_env.sh '{VENV_DIR}'",
    "00_setup_env.log",
    use_venv=False,
)
run_logged("python -m unittest discover -s ../tests -v", "00_unit_tests.log")

## Step 4A：Sub-demo 2A 合成闭环验证（MultiVI + scDoRI 塌缩审计 + DotPlot）

In [ ]:
run_logged(
    "python scripts/01_prepare_multiome_mudata.py --source synthetic && "
    "python scripts/02_run_multivi_baseline.py --synthetic && "
    "python scripts/04_sanity_check_compare.py --synthetic",
    "01_sub2a_synthetic.log",
    cwd=SUB2A_ROOT,
)
backup_subdemo("subdemo_2a_multiome_egrn")

In [ ]:
# 预览 Sub-demo 2A 验收图表与摘要
import json
from IPython.display import Image, display

summary_p = SUB2A_ROOT / "outputs/tables/sanity_summary.json"
if summary_p.exists():
    print(json.dumps(json.loads(summary_p.read_text(encoding="utf-8")), indent=2))
for fig_name in ["umap_multivi_vs_scdori.png", "topic_activity_diagnostics.png", "egrn_consistency_dotplot.png"]:
    p = SUB2A_ROOT / "outputs/figures" / fig_name
    if p.exists():
        display(Image(filename=str(p)))

## Step 4B：Sub-demo 2B 合成闭环验证（Squidpy + cell2location + Niche + DPT 伪时间投影）

In [ ]:
run_logged(
    "python scripts/01_download_spatial_and_ref.py --source synthetic && "
    "python scripts/02_spatial_qc_and_squidpy.py --synthetic && "
    "python scripts/03_run_cell2location_deconv.py --synthetic && "
    "python scripts/04_spatial_niches_and_trajectory.py",
    "02_sub2b_synthetic.log",
    cwd=SUB2B_ROOT,
)
backup_subdemo("subdemo_2b_spatial_trajectory")

In [ ]:
# 预览 Sub-demo 2B H&E 叠加图与伪时间空间投影
from IPython.display import Image, display

for fig_name in [
    "squidpy_spatial_markers.png",
    "he_overlay_cell_abundances.png",
    "he_overlay_spatial_niches.png",
    "spatial_pseudotime_trajectory.png",
]:
    p = SUB2B_ROOT / "outputs/figures" / fig_name
    if p.exists():
        display(Image(filename=str(p)))

## Step 5A：真实公开数据 — Sub-demo 2A（10x PBMC 3k/10k Multiome + 官方 `scDoRI` Notebook）
先运行 PBMC 3k 跑通 `MultiVI` + 官方 `bioFAM/scDoRI` 三阶段（预处理 → 双阶段训练 → 下游 eGRN）；若已有 GBM `E-MTAB-17183` `.h5ad` 可在下方改用 `--source h5ad` 与 `configs/scdori_gbm_hg38.json`。

In [ ]:
# 可选：断线恢复
# restore_subdemo("subdemo_2a_multiome_egrn")

run_logged(
    "python scripts/01_prepare_multiome_mudata.py --config configs/pbmc_demo.yaml --source pbmc3k && "
    "python scripts/02_run_multivi_baseline.py --config configs/pbmc_demo.yaml",
    "03_sub2a_pbmc_multivi.log",
    cwd=SUB2A_ROOT,
)
backup_subdemo("subdemo_2a_multiome_egrn")

run_logged(
    "bash scripts/03_run_scdori.sh --config configs/scdori_pbmc_hg38.json --stage all --kernel meddemo_dir2_py311 && "
    "python scripts/04_sanity_check_compare.py --config configs/pbmc_demo.yaml --scdori-config configs/scdori_pbmc_hg38.json",
    "04_sub2a_pbmc_scdori.log",
    cwd=SUB2A_ROOT,
)
backup_subdemo("subdemo_2a_multiome_egrn")

## Step 5B：真实公开数据 — Sub-demo 2B（`MahmoodLab/hest` 或免鉴权 `10x Human GBM` Visium + 单细胞参考集）
- 若已同意 `https://huggingface.co/datasets/MahmoodLab/hest` 使用条款，可填入 `HF_TOKEN` 并将自己的单细胞参考集（如 `Neftel SCP393` / `GBmap` 的 `.h5ad`）路径填到 `REF_H5AD_PATH`；
- **若留空或 Drive 上不存在参考集文件，脚本会自动回退到免鉴权公开的 10x Genomics Human Glioblastoma Visium (`Parent_Visium_Human_Glioblastoma`) + 匹配 GBM 单细胞参考集 (`Parent_SC3v3_Human_Glioblastoma`) 直接跑通全流程。**

In [ ]:
HF_TOKEN = ""  # 可选：留空则自动回退到免鉴权 10x Human GBM 数据集
REF_H5AD_PATH = "/content/drive/MyDrive/MedDemo_inputs/gbmap_or_neftel_raw.h5ad"

if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    from huggingface_hub import login
    login(token=HF_TOKEN)

# 可选：断线恢复
# restore_subdemo("subdemo_2b_spatial_trajectory")

run_logged(
    f"python scripts/01_download_spatial_and_ref.py --config configs/gbm_spatial_demo.yaml --source hest --ref-h5ad '{REF_H5AD_PATH}' && "
    "python scripts/02_spatial_qc_and_squidpy.py --config configs/gbm_spatial_demo.yaml && "
    "python scripts/03_run_cell2location_deconv.py --config configs/gbm_spatial_demo.yaml && "
    "python scripts/04_spatial_niches_and_trajectory.py --config configs/gbm_spatial_demo.yaml",
    "05_sub2b_hest_e2e.log",
    cwd=SUB2B_ROOT,
)
backup_subdemo("subdemo_2b_spatial_trajectory")